# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1805shahab/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
!pip -q install duckdb

import duckdb
import pandas as pd

# Replace with your own Hugging Face Read token
HF_TOKEN = "hugging face token"

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN """hugging face token"""
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5000
""").df()

print(df.shape)
display(df.head())

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

feature_df = df[feature_columns].copy()

# Fill missing values
feature_df = feature_df.fillna(0)

print("Feature vector shape:", feature_df.shape)
display(feature_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(5000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


Feature vector shape: (5000, 5)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,0,0
1,1,0,0.000000,0,0
2,125,1,4.928000,0,0
3,7,0,4.000000,0,0
4,11,0,2.272727,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
"""# Feature Notes

| Feature | Meaning | Missing Value Handling | Available Before Prediction? |
|----------|---------|------------------------|------------------------------|
| gsc_impressions | Number of Google Search impressions | Filled with 0 | Yes |
| gsc_clicks | Historical Google Search clicks | Filled with 0 | Yes |
| gsc_avg_position | Average Google Search position | Filled with 0 | Yes |
| ga4_pageviews | Historical page views from GA4 | Filled with 0 | Yes |
| ga4_sessions | Historical sessions from GA4 | Filled with 0 | Yes |

All selected features are historical observations that exist before making a content refresh decision."""

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create a simple demonstration label
demo_df = feature_df.copy()
demo_df["label"] = (df["gsc_clicks"] == 0).astype(int)

# Honest model
X = demo_df.drop(columns=["label"])
y = demo_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, pred)

print("Accuracy without leakage:", round(honest_accuracy, 4))

# ----------------------------
# Deliberate leakage
# ----------------------------

X_leak = X.copy()
X_leak["leak_feature"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leak_accuracy = accuracy_score(y_test, pred)

print("Accuracy with leakage:", round(leak_accuracy, 4))

# Remove the leaking feature
X_leak = X_leak.drop(columns=["leak_feature"])

print("\nLeak feature removed.")

Accuracy without leakage: 1.0
Accuracy with leakage: 1.0

Leak feature removed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
"""# Excluded Fields

| Field | Reason |
|--------|--------|
| client_hash_id | Identifier only; does not represent page performance. |
| content_hash_id | Unique identifier with no predictive meaning. |
| report_date | Used for ordering observations, not as a model feature. |
| month | Dataset partition field rather than a predictive signal. |
| Future clicks or pageviews | Not available at prediction time and would introduce leakage. |
| Future engagement metrics | Reveal future outcomes and artificially inflate model performance. |"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.